# Multimodal Cancer Classification Challenge 2026 — v12

**v11 LB = 0.6479 (up from v10's 0.5846).** OOF patient-level AUC was 1.0, OOF cell-level was 0.867 — but LB is 0.65, a 0.22 gap. That gap is dominated by **domain shift**: test patients have different stain intensity, microscope settings, or exposure than the 12 train patients. v12 targets that directly.

**Changes vs v11:**

1. **Per-image normalization** instead of dataset-level mean/std. Each cell image is normalized to zero-mean unit-std using its own statistics. This wipes out patient-level intensity baselines (the strongest spurious feature on small cohorts).
2. **Stronger color/intensity augmentation**: ColorJitter brightness=0.5, contrast=0.5; random gamma γ ∈ [0.7, 1.4]; random Gaussian noise σ=0.01 with p=0.3.
3. **CutMix (α=1.0)** instead of Mixup. Preserves local cell morphology better than alpha-blending entire images.
4. **Stochastic Depth** (`drop_path_rate=0.1`) via timm ResNet-18. Better regularizer than plain dropout for residual nets.
5. **AUC-gated ensemble at submission time**: only include checkpoints with `val_auc ≥ 0.78`. The full-data model is always included.

**Kept from v11 (because they worked):**
- RAM-cached JPEG bytes (eliminates I/O bottleneck)
- Patient-balanced sampler (each batch sees 4 distinct patients)
- 3 folds × 2 seeds = 6 CV models + 1 full-data model
- 8-way D4 TTA at inference
- ResNet-18 backbone (22 M params)
- BCE + pos_weight, OneCycleLR, AMP, gradient clipping

**Runtime budget:** ~4 h total on T4 (same as v11 — augmentations are CPU-side, model unchanged).

**Kaggle settings:** Accelerator = GPU T4 x2 (only GPU 0 used); Internet On; Persistence Files only.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
print("timm:", timm.__version__)
!nvidia-smi -L

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/rafaelproena/a3-adl")
assert (DATA_ROOT / "train.csv").exists(), f"train.csv not at {DATA_ROOT}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# CV
N_SPLITS    = 3
BASE_SEED   = 1
SEEDS       = [1, 2]

# Optimization
EPOCHS      = 10
PATIENCE    = 5
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0

# Regularization
CUTMIX_ALPHA  = 1.0                 # CutMix replaces Mixup
CUTMIX_PROB   = 0.5                 # apply CutMix on 50% of batches
DROPOUT       = 0.3
DROP_PATH     = 0.1                 # Stochastic Depth in ResNet via timm

# Augmentation
COLOR_BRIGHTNESS = 0.5              # was 0.2 in v11
COLOR_CONTRAST   = 0.5              # was 0.2 in v11
GAMMA_RANGE      = (0.7, 1.4)
NOISE_SIGMA      = 0.01
NOISE_PROB       = 0.3

# Model
PRETRAINED   = True

# Sampler
NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4              # 4 patients × 32 cells = 128 per batch

# Full-data model
TRAIN_FULL_DATA_MODEL = True

# Ensemble gating at submission time
ENSEMBLE_MIN_AUC = 0.78              # drop fold-models below this val_auc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(BASE_SEED)

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf

    def __len__(self): return len(self.df)

    @staticmethod
    def _decode(buf):
        return Image.open(io.BytesIO(buf)).convert("L")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

In [ ]:
def stratified_patient_kfold(df, n_splits=3, seed=1):
    skgf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = df["Diagnosis"].to_numpy(); groups = df["patient_id"].to_numpy()
    splits = list(skgf.split(df, y=y, groups=groups))
    for f, (_, va) in enumerate(splits):
        if len(np.unique(y[va])) < 2:
            raise ValueError(f"Fold {f} has only one class — try another seed.")
    return splits

def summarize_split(df, tr, va):
    trd, vad = df.iloc[tr], df.iloc[va]
    return (f"train: {len(tr):>6} cells, {trd['patient_id'].nunique():>2}p, "
            f"pos {trd['Diagnosis'].mean():.3f} | "
            f"val: {len(va):>6} cells, {vad['patient_id'].nunique():>2}p, "
            f"pos {vad['Diagnosis'].mean():.3f} | "
            f"val pats: {sorted(vad['patient_id'].unique().tolist())}")

class PatientBalancedSampler(Sampler):
    """Each batch has cells from `patients_per_batch` distinct patients in equal proportions."""
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size

    def __len__(self): return self.epoch_len

    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
# Per-image normalization: each image is normalized to zero mean / unit std using
# its OWN pixel statistics. This wipes out patient-level intensity baselines,
# which were our top suspect for the OOF→LB gap.
EPS = 1e-6

def per_image_normalize(img_tensor):
    # img_tensor: (1, H, W) float in [0, 1]
    mean = img_tensor.mean()
    std  = img_tensor.std().clamp(min=EPS)
    return (img_tensor - mean) / std

def to_tensor_pernorm(img):
    return per_image_normalize(TF.to_tensor(img))


class RandomGamma:
    """Apply random gamma correction *before* normalization (input must be in [0,1])."""
    def __init__(self, low=0.7, high=1.4, p=0.7):
        self.low = low; self.high = high; self.p = p
    def __call__(self, img):  # PIL image
        if random.random() < self.p:
            g = random.uniform(self.low, self.high)
            img = TF.adjust_gamma(img, g)
        return img

class AddGaussianNoise:
    """Apply Gaussian noise AFTER normalization (works on tensors)."""
    def __init__(self, sigma=0.01, p=0.3):
        self.sigma = sigma; self.p = p
    def __call__(self, t):
        if random.random() < self.p:
            t = t + torch.randn_like(t) * self.sigma
        return t


class PairedGeoAug:
    """Same geometric transform on both BF and FL: D4 + mild rotation."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=15.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot

    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        return bf, fl


def train_modality_transform(modality):
    # modality is informational; per-image normalization makes modality stats irrelevant.
    return T.Compose([
        T.ColorJitter(brightness=COLOR_BRIGHTNESS, contrast=COLOR_CONTRAST),
        RandomGamma(*GAMMA_RANGE, p=0.7),
        to_tensor_pernorm,
        AddGaussianNoise(sigma=NOISE_SIGMA, p=NOISE_PROB),
    ])

def eval_modality_transform(modality):
    return to_tensor_pernorm

In [ ]:
def _make_resnet18_branch(pretrained=True, drop_path_rate=DROP_PATH):
    """timm ResNet-18 with Stochastic Depth, 1-channel input."""
    net = timm.create_model("resnet18", pretrained=pretrained, num_classes=0,
                            global_pool="avg", drop_path_rate=drop_path_rate,
                            in_chans=1)
    return net, net.num_features  # 512

class MultimodalClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=DROPOUT, drop_path=DROP_PATH):
        super().__init__()
        self.bf_branch, fd = _make_resnet18_branch(pretrained, drop_path)
        self.fl_branch, _  = _make_resnet18_branch(pretrained, drop_path)
        self.head = nn.Sequential(
            nn.Linear(fd * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    print("Output shape:", _m(_x, _x).shape,
          "  params:", sum(p.numel() for p in _m.parameters()) // 1_000_000, "M")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

In [ ]:
def cutmix_batch(bf, fl, y, alpha=1.0):
    """CutMix: cut a random rectangle from another sample and paste into the current
    sample, in both modalities (same coordinates so they stay aligned).
    Returns mixed images and a soft label.
    """
    if alpha <= 0:
        return bf, fl, y
    lam = float(np.random.beta(alpha, alpha))
    B, C, H, W = bf.shape
    idx = torch.randperm(B, device=bf.device)
    # rectangle size proportional to sqrt(1-lam)
    cut_ratio = (1.0 - lam) ** 0.5
    cut_h, cut_w = int(H * cut_ratio), int(W * cut_ratio)
    if cut_h == 0 or cut_w == 0:
        return bf, fl, y
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1 = max(0, cy - cut_h // 2); y2 = min(H, cy + cut_h // 2)
    x1 = max(0, cx - cut_w // 2); x2 = min(W, cx + cut_w // 2)
    bf_out, fl_out = bf.clone(), fl.clone()
    bf_out[:, :, y1:y2, x1:x2] = bf[idx, :, y1:y2, x1:x2]
    fl_out[:, :, y1:y2, x1:x2] = fl[idx, :, y1:y2, x1:x2]
    # Actual lam from area ratio
    lam_eff = 1.0 - ((y2 - y1) * (x2 - x1)) / (H * W)
    y_mixed = lam_eff * y + (1 - lam_eff) * y[idx]
    return bf_out, fl_out, y_mixed

In [ ]:
def run_epoch(model, loader, optimizer, scaler, criterion, train,
              cutmix_alpha=0.0, cutmix_prob=0.0, grad_clip=0.0, sched=None,
              log_every=0):
    model.train(train)
    losses, hard_ys, ps = [], [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())

        if train and cutmix_alpha > 0 and random.random() < cutmix_prob:
            bf, fl, y = cutmix_batch(bf, fl, y, cutmix_alpha)

        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss = criterion(logits, y)

        if train:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if sched is not None and scaler.get_scale() >= old_scale:
                    sched.step()
            else:
                loss.backward()
                if grad_clip > 0: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if sched is not None: sched.step()

        losses.append(loss.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())

        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | loss {float(np.mean(losses[-log_every:])):.4f}")

    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc, hard_ys, ps


def train_one_model(train_df, val_df, ckpt_path, oof_path, hist_path,
                   epochs, seed, sampler_kind="patient", track_oof=True):
    seed_everything(seed)

    train_ds = CachedCellDataset(train_df, bf_train_cache, fl_train_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=PairedGeoAug())
    if sampler_kind == "patient":
        sampler = PatientBalancedSampler(train_df, batch_size=BATCH_SIZE,
                                         patients_per_batch=PATIENTS_PER_BATCH, seed=seed)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    val_loader = None
    if val_df is not None:
        val_ds = CachedCellDataset(val_df, bf_train_cache, fl_train_cache,
                                   eval_modality_transform("bf"),
                                   eval_modality_transform("fl"))
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

    model = MultimodalClassifier(pretrained=PRETRAINED, dropout=DROPOUT, drop_path=DROP_PATH).to(DEVICE)
    pos = (train_df["Diagnosis"] == 1).sum()
    neg = (train_df["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  seed={seed}  epochs={epochs}  drop_path={DROP_PATH}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR,
        steps_per_epoch=len(train_loader), epochs=epochs, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    history, best_auc, best_ep, no_improve = [], -1.0, 0, 0
    for ep in range(epochs):
        t0 = time.time()
        tr_loss, tr_auc, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, criterion, True,
            cutmix_alpha=CUTMIX_ALPHA, cutmix_prob=CUTMIX_PROB,
            grad_clip=GRAD_CLIP, sched=sched, log_every=200)
        va_loss = va_auc = float("nan"); vy = vp = None
        if val_loader is not None:
            with torch.no_grad():
                va_loss, va_auc, vy, vp = run_epoch(
                    model, val_loader, None, None, criterion, False)
        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} "
              f"| va_loss {va_loss:.4f} va_auc {va_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc,
                        "va_loss": va_loss, "va_auc": va_auc, "time": dt})

        save_now = False
        if val_loader is not None:
            if va_auc > best_auc:
                best_auc, best_ep, no_improve = va_auc, ep, 0; save_now = True
            else:
                no_improve += 1
        else:
            save_now = True; best_ep = ep

        if save_now:
            torch.save({"model": model.state_dict(), "epoch": ep,
                        "val_auc": va_auc if val_loader is not None else None,
                        "args": {"dropout": DROPOUT, "drop_path": DROP_PATH}}, ckpt_path)
            if track_oof and val_loader is not None:
                pd.DataFrame({"Name": val_df["Name"].values,
                              "patient_id": val_df["patient_id"].values,
                              "y_true": vy, "y_pred": vp}).to_csv(oof_path, index=False)

        if val_loader is not None and no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {ep}"); break

    with open(hist_path, "w") as f:
        json.dump({"history": history, "best_auc": best_auc, "best_ep": best_ep}, f, indent=2)

    del model, optimizer, sched, scaler, train_loader
    if val_loader is not None: del val_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_auc, best_ep

In [ ]:
splits = stratified_patient_kfold(df_train, N_SPLITS, seed=BASE_SEED)
print(f"CV folds={len(splits)}  seeds={SEEDS}\n")

all_results = []
for fold, (tr, va) in enumerate(splits):
    print(f"=== FOLD {fold} ===")
    print("  " + summarize_split(df_train, tr, va))
    train_df = df_train.iloc[tr].reset_index(drop=True)
    val_df   = df_train.iloc[va].reset_index(drop=True)

    for seed in SEEDS:
        tag = f"fold{fold}_seed{seed}"
        print(f"  --- {tag} ---")
        ckpt = OUT_DIR / f"{tag}_best.pt"
        oof  = OUT_DIR / f"{tag}_oof.csv"
        hist = OUT_DIR / f"{tag}_history.json"
        best_auc, best_ep = train_one_model(
            train_df, val_df, ckpt, oof, hist, epochs=EPOCHS, seed=seed)
        all_results.append({"fold": fold, "seed": seed,
                            "best_auc": best_auc, "best_ep": best_ep})
        print(f"  {tag}: best AUC = {best_auc:.4f} at ep {best_ep}\n")

df_results = pd.DataFrame(all_results)
print("\n=== CV summary ===")
print(df_results.to_string(index=False))
print(f"Mean best AUC: {df_results['best_auc'].mean():.4f}  std {df_results['best_auc'].std():.4f}")
print(f"Median best epoch: {int(df_results['best_ep'].median())}")

In [ ]:
if TRAIN_FULL_DATA_MODEL:
    median_ep = int(df_results['best_ep'].median()) + 1
    full_epochs = max(median_ep, 5)
    print(f"\n=== FULL-DATA MODEL (epochs={full_epochs}) ===")
    ckpt = OUT_DIR / "fulldata_best.pt"
    hist = OUT_DIR / "fulldata_history.json"
    train_one_model(df_train, None, ckpt, None, hist,
                    epochs=full_epochs, seed=BASE_SEED + 100,
                    sampler_kind="patient", track_oof=False)
    print("  full-data model saved.")
else:
    print("(skipping full-data model)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for hp in sorted(glob.glob(str(OUT_DIR / "fold*_history.json"))):
    h = json.load(open(hp))["history"]
    label = Path(hp).stem.replace("_history", "")
    ax[0].plot([e["epoch"] for e in h], [e["va_loss"] for e in h], marker="o", label=label)
    ax[1].plot([e["epoch"] for e in h], [e["va_auc"]  for e in h], marker="o", label=label)
ax[0].set(title="Validation loss", xlabel="epoch", ylabel="BCE")
ax[1].set(title="Validation AUC",  xlabel="epoch", ylabel="AUC")
ax[1].axhline(0.85, color="red", linestyle="--", alpha=0.5, label="target 0.85")
for a in ax: a.legend(fontsize=8); a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fold_oofs = []
for fold in range(N_SPLITS):
    seed_oofs = []
    for seed in SEEDS:
        p = OUT_DIR / f"fold{fold}_seed{seed}_oof.csv"
        if p.exists(): seed_oofs.append(pd.read_csv(p))
    if not seed_oofs: continue
    base = seed_oofs[0][["Name", "patient_id", "y_true"]].copy()
    base["y_pred"] = np.mean([d["y_pred"].values for d in seed_oofs], axis=0)
    fold_oofs.append(base)
if fold_oofs:
    oof = pd.concat(fold_oofs, ignore_index=True)
    cell_auc = roc_auc_score(oof["y_true"], oof["y_pred"])
    pp = oof.groupby("patient_id").agg(
        mean_pred=("y_pred", "mean"), median_pred=("y_pred", "median"),
        label=("y_true", "first")).sort_values("mean_pred")
    pat_auc = roc_auc_score(pp["label"], pp["mean_pred"])
    print("=== OOF (seed-averaged) ===")
    print(pp.to_string())
    print(f"\ncell-level OOF AUC: {cell_auc:.4f}    patient-level AUC: {pat_auc:.4f}")

In [ ]:
def _d4(bf, fl):
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def load_model_from_ckpt(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    drop_path = float(args.get("drop_path", DROP_PATH))
    model = MultimodalClassifier(pretrained=False, dropout=dropout, drop_path=drop_path).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict_one_ckpt(ckpt_path, loader, tta=True):
    model = load_model_from_ckpt(ckpt_path)
    preds = []
    n_aug = 8 if tta else 1
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in (_d4(bf, fl) if tta else [(bf, fl)]):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# AUC-gated ensemble: keep CV models with val_auc >= ENSEMBLE_MIN_AUC,
# always keep the full-data model (no val_auc to compare).
ckpts_all = sorted(glob.glob(str(OUT_DIR / "fold*_best.pt")))
ckpts = []
for c in ckpts_all:
    s = torch.load(c, map_location="cpu", weights_only=False)
    va = s.get("val_auc")
    if va is not None and va >= ENSEMBLE_MIN_AUC:
        ckpts.append(c); print(f"  KEEP {Path(c).name}  val_auc={va:.4f}")
    else:
        print(f"  DROP {Path(c).name}  val_auc={va}")
if TRAIN_FULL_DATA_MODEL and (OUT_DIR / "fulldata_best.pt").exists():
    ckpts.append(str(OUT_DIR / "fulldata_best.pt"))
    print(f"  KEEP fulldata_best.pt (always included)")

if not ckpts:
    raise RuntimeError(f"No checkpoints passed the gate at {ENSEMBLE_MIN_AUC}")

all_preds = []
for c in ckpts:
    t0 = time.time()
    all_preds.append(predict_one_ckpt(c, test_loader, tta=True))
    print(f"  {Path(c).name} done in {time.time()-t0:.1f}s")
preds = np.mean(all_preds, axis=0)

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.3f}, "
      f"min {preds.min():.3f}, max {preds.max():.3f})")
print(sub.head())
!wc -l /kaggle/working/submission.csv